# Unit 5 – Practical Modelling and Predictions

Today we will learn how to practically tune a classifier by using cross-validation and a hold-out test set. Finally, there will be a competition for you to hone your skills.

Let's dive right into it. 

## Today's idioms
### Training a naive bayes classifier in Scikit-learn
Scikit-learn comes with various implementations of the Naive Bayes classifier. The "standard" one is the Gaussian variant, which assumes the data has a normal distribution (although it works reasonably well for cases where this assumption doesn't quite hold).

In [ ]:
import matplotlib
%matplotlib inline
import numpy as np
import pandas as pd
import sklearn.datasets as data

X, y  = data.load_iris(return_X_y=True)  # returns data to be used in estimator
# X is the data, y are the targets:
print("data has shape {}, targets have shape {}.".format(X.shape, y.shape))


We instantiate and train/fit the classifier:

In [ ]:
import sklearn.naive_bayes as nb
gnb = nb.GaussianNB() # instantiates a Gaussian Naive Bayes object
gnb.fit(X=X, y=y)

Let's predict the training data:

In [ ]:
y_pred = gnb.predict(X=X)
y_pred

Note that the targets 0, 1, 2 stand for *setosa*, *virginica* and *versicolor*. It looks like there are a few errors though! This is how the real targets look:

In [ ]:
y

Let's compare the two:

In [ ]:
y - y_pred

This would be all zeros if the prediction was correct. Let's compute the accuracy, i.e. the percetage of correct predictions:

In [ ]:
y != y_pred

This is a more general way to compare classifications. Think about when these two ways produce different results and why this one is correct.

In [ ]:
fraction_wrong = np.sum(y != y_pred)/len(y) # how many nonzeros, divided by total number
acc = 100*(1-fraction_wrong)
print("accuracy is {:.2f}%".format(acc))

96 percent correct!

let's see what an SVM can do.

### Training an SVM in scikit-learn

We use a linear SVM with a linear kernel. The process is the standard one. First we instantiate the classifier:

In [ ]:
from sklearn import svm
svc = svm.SVC(kernel='linear') 

Then we fit and predict: 

In [ ]:
svc.fit(X=X, y=y)
y_pred = svc.predict(X=X)

Finally we compute the accuracy:

In [ ]:
acc = 100*(1-np.sum(y != y_pred)/len(y))
print("accuracy is {:.2f}%".format(acc))

hey, that's already substantially better than Naive Bayes! Later you can try to get even better by tuning the parameters.

But first let's try a Multi-Layer Neural Network.

### Training a Multi-Layer Neural Network in scikit-learn

By now you should have gotten the gist of how to instantiate, train and predict using a classifier. The process is standard in `sklearn`, that means you can use it with any estimator/classifier. 

Here we lump it all together in one cell. 

In [ ]:
from sklearn import neural_network as nn
mlp = nn.MLPClassifier()  #instantiates a Multi-Layer Perceptron (a specific Multi-Layer Neural Network)
mlp.fit(X=X, y=y)
y_pred = mlp.predict(X=X)
acc = 100*(1-np.sum(y != y_pred)/len(y))
print("accuracy is {:.2f}%".format(acc))


Oh, a warning! It complains that the training process of the network has not yet converged, because the maximum iterations the algorithm is instructed to perform (by default) have been reached. 

### Task 1 - fix the warning
Check the documentation of `sklearn.neural_network.MLPClassifier` to find the parameter that controls the maximum number of iterations. Set it to a higher value until the warning goes away.

In [ ]:
nn.MLPClassifier?

Fix the code below to make the warning disappear:

In [ ]:
mlp = nn.MLPClassifier()  #here you need to add the parameter
mlp.fit(X=X, y=y)
y_pred = mlp.predict(X=X)
acc = 100*(1-np.sum(y != y_pred)/len(y))
print("accuracy is {:.2f}%".format(acc))

## Cross-validation 

Up to now we trained the classifier on all available data. It is very likely that we are overestimating the ability of the trained classifier to predict unseen data. But since we don't *have* unseen data, how can we test how well the classifier might generalise?

The solution is **Cross-validation**. In a nutshell, it means you split your data into several equal-sized chunks, so-called folds. You keep one fold out and train the classifier on the remaining chunks. Repeat until each fold has been held out once. Then compute the accuracy on the held-out folds. 

The scikit-learn online documentation has a [good overview](https://scikit-learn.org/stable/modules/cross_validation.html) I recommend to read should you feel you need a refresher.  

The cross-calidation module is straightforward to use. It does the data splitting for you and loops over the fit/predict cycle. All you need to provide is an instance of a classifier.

Below is an example of cross-validation with an SVM with a linear kernel and `C=1`:

In [ ]:
import sklearn.model_selection as ms
clf = svm.SVC(kernel='linear', C=1)
scores = ms.cross_val_score(clf, X, y, cv=5)
scores

Let's compute the mean score and the 95% confidence interval (i.e., two standard deviations):

In [ ]:
print("Accuracy: %0.2f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))

## Parameter tuning: some examples

### Creating a hold-out data set: test-training split
When tuning parameters, it is advisable to create a hold-out set that you don't touch until your parameter tuning campaign is finished. 

You can create multiple candidate tuning sets and use the hold-out set at the very end in order to find the model that generalises best.

Scikit-learn provides the `test_train_split` function that provides this functionality. The below example creates a 40% test/training split:

In [ ]:
X_train, X_test, y_train, y_test = ms.train_test_split(
     X, y, test_size=0.4, random_state=0)

Let's look at the sizes:

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
X_test.shape

In [ ]:
y_test.shape

That looks fine. 

In the remainder, we will not be touching the test set until we have completed the parameter optimisation. 

### SVM tuning

Tunable parameters for soft-margin SVMs include:
 * $C$, a positive, numerical value, that controls the penalty for misclassifications.
 * the kernel choice. Sensible beginner choices in scikit-learn are `linear`, `poly`, `rbf`, `sigmoid`. These come with different tuning options. 

#### Stuck? Read the docs!
Refer to the documentation on [support vector classification](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC) for more info.

There is also an excellent [SVM user guide](https://scikit-learn.org/stable/modules/svm.html) available in the on-line documentation of scikit-learn. Read 

### Task 2: Hands-on tuning
Use the code below to try out a few parameter settings. Note that it uses cross-validation, using the previously generated training data from the `train_test_split` above.

In [ ]:
svc = svm.SVC(kernel='linear', C=1)
scores = ms.cross_val_score(svc, X_train, y_train, cv=5)
print("Accuracy: %0.4f (+/- %0.4f)" % (scores.mean(), scores.std() * 2))

In [ ]:
## Attempt 1

In [ ]:
## Attempt 2

In [ ]:
## Attempt 3

In [ ]:
## Attempt ..

When you have found a good combination of $C$ and kernel choice, execute the code below to store that classifier in the variable `best_svm`, and proceed to the next part, "Neural Networks"

In [ ]:
best_svm = svc

### Neural networks

Neural networks have tons of parameters to tune, and they are usually much more difficult to understand than SVM's parameters. Have a look at the [MLPClassifier on-line documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html#sklearn.neural_network.MLPClassifier) to get a glimpse. Note that these are not even deep networks, for which the number of parameters multiply!

The most common tuning parameters are.
* `hidden_layer_sizes`: the number of the number of hidden layers and the neurons therein. Examples:
    * `hidden_layer_sizes=(100,)` would create a single hidden layer with 100 neurons (the default)
    * `hidden_layer_sizes=(20,10)` would create two hidden layers, the first with 20 and the second with 10 neurons. 
* `activation`: the transfer function. The default is `relu`, the "REctifying Linear Unit", but other choices are worth a try, e.g. the signmoidal functions `sigmoid`, `tanh`, or `logistic`.
* `learning_rate`: The default is constant, but for complicated data sets it can help to set it to other options. Check the documentation for an explanation. 

The [MLPClassifier on-line documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html#sklearn.neural_network.MLPClassifier) states a useful tweak to the `solver` parameter:
> The default solver ‘adam’ works pretty well on relatively large datasets (with thousands of training samples or more) in terms of both training time and validation score. For small datasets, however, ‘lbfgs’ can converge faster and perform better.

We will therefore use the 'lbfgs' solver. 

* best MLP should be stored in variable `best_mlp`

In [ ]:
mlp = nn.MLPClassifier(hidden_layer_sizes=(100,10), 
                       activation='relu',
                       learning_rate='constant',
                       solver='lbfgs')
scores = ms.cross_val_score(mlp, X_train, y_train, cv=5)
print("Accuracy: %0.4f (+/- %0.4f)" % (scores.mean(), scores.std() * 2))

As above we store the best network for evaluation of the test set. 

In [ ]:
best_mlp = mlp

### What's the best classfier?
Now our hold-out dataset comes into play. Let's evaluate the best MLP and the best SVM on the hold-out set.

Verify that the best_svm has the correct parameters:

In [ ]:
best_svm

We train it on the entire training set and evaluate it on the held-out test set.

In [ ]:
# SVM 
best_svm.fit(X=X_train, y=y_train)
y_pred_svm = best_svm.predict(X_test)
acc = 100*(1-np.sum(y_test != y_pred_svm)/len(y_test))
print("accuracy is {:.4f}%".format(acc))



Same for the Neural Network:

In [ ]:
best_mlp

In [ ]:
# MLP 
best_mlp.fit(X=X_train, y=y_train)
y_pred_mlp = best_mlp.predict(X_test)
acc = 100*(1-np.sum(y_test != y_pred_mlp)/len(y_test))
print("accuracy is {:.4f}%".format(acc))



### Task 3 - Repeat with Naïve Bayes classifier
Repeat the training procedure with the Naive Bayes algorithm. Copy necessary code from above and adapt to use the training set for training and the testing set for prediction.

In [ ]:
## enter your code here.

### Task 4 - Comparision and Understanding
How does the Naive Bayes classifier compare to the MLP and SVM classifiers?
In this worksheet we only looked at accuracy, what else could we look at?
The Naive Bayes does not have real options to change its outcome like the MLP and SVM, why is this?

In [ ]:
## convert to a markdown cell and write your answer